# 04 — OCT Model : ResNet50 sur images B-scan

**Objectif :** Entraîner un ResNet50 (préentraîné ImageNet) sur les images OCT B-scan avec validation croisée K=5.

## Architecture
```
ResNet50 (préentraîné ImageNet)
    ├── Backbone (couches gelées → phase A, dégelées → phase B)
    └── Tête custom :
           ├── Linear(2048 → 512)
           ├── BatchNorm1d(512) + ReLU + Dropout(0.4)
           └── Linear(512 → 1)   ← logit binaire (BCEWithLogitsLoss)
```

## Stratégie 2-phases (identique Fundus)
```
Phase A (epochs 1-5) : backbone gelé → seule la tête s'entraîne
Phase B (epoch 6+)   : backbone dégelé → fine-tuning complet
```

## Différences clés vs Notebook 03 (Fundus)
| Aspect | Fundus | OCT |
|--------|--------|-----|
| Chargement image | RGB direct | Grayscale → RGB |
| Augmentation couleur | HueSaturationValue | ❌ (supprimé) |
| CLAHE | clip=2.0, p=0.3 | clip=3.0, p=0.4 (plus fort) |
| Rotation | ±30° | ±10° (structures horizontales) |
| CoarseDropout holes | 8×20px | 6×40px (occlusion horizontale) |
| Dataset class | FundusDataset | OCTDataset |
| Sortie .npy | fundus_probabilities.npy | oct_probabilities.npy |

## Étapes
| Étape | Description |
|-------|-------------|
| 1 | Reconstruction de df_pairs (identique à 01_eda) |
| 2 | OCTDataset + Transforms Albumentations OCT |
| 3 | StratifiedKFold (K=5) |
| 4 | Boucle d'entraînement avec early stopping |
| 5 | Courbes loss/accuracy par fold |
| 6 | Métriques : Accuracy, F1, ROC-AUC, Sensibilité, Spécificité |
| 7 | Sauvegarde oct_model_fold{k}.pth + oct_probabilities.npy |

## Section 1 : Imports

In [ ]:
# ── Standard Library ───────────────────────────────────────────────────────
import os
import json
import time
import copy
import warnings
from pathlib import Path

# ── Data ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from PIL import Image
import cv2

# ── PyTorch ────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

# ── Albumentations ─────────────────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Sklearn ────────────────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, f1_score, accuracy_score
)

# ── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444',
    'axes.labelcolor': '#e0e0e0',
    'axes.titlecolor': '#ffffff',
    'xtick.color': '#aaaaaa',
    'ytick.color': '#aaaaaa',
    'text.color': '#e0e0e0',
    'grid.color': '#333355',
    'grid.linestyle': '--',
    'grid.alpha': 0.4,
    'font.size': 11,
})

PALETTE = {
    'glaucoma': '#e74c3c',
    'normal':   '#2ecc71',
    'train':    '#3498db',
    'val':      '#f39c12',
    'neutral':  '#9b59b6',
}

print('✅ Imports OK')
print(f'   PyTorch        : {torch.__version__}')
print(f'   Albumentations : {A.__version__}')
print(f'   Device         : {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU            : {torch.cuda.get_device_name(0)}')

## Section 2 : Configuration

In [ ]:
# ── Chemins ────────────────────────────────────────────────────────────────
DATA_ROOT  = '/kaggle/input/datasets/marwanenasri/data-on-oct-and-fundus-images-zip/Data on OCT and Fundus Images'
EXCEL_PATH = os.path.join(DATA_ROOT, 'Patient Record-final.xlsx')
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Hyperparamètres ────────────────────────────────────────────────────────
# Identiques au notebook 03 pour cohérence
CFG = {
    'img_size'      : 224,
    'batch_size'    : 8,
    'n_folds'       : 5,
    'seed'          : 42,
    'num_epochs'    : 50,
    'learning_rate' : 1e-4,
    'backbone_lr'   : 1e-5,
    'weight_decay'  : 1e-4,
    'patience'      : 10,
    'lr_patience'   : 5,
    'lr_factor'     : 0.5,
    'unfreeze_epoch': 5,
    'num_workers'   : 2,
    'imagenet_mean' : [0.485, 0.456, 0.406],
    'imagenet_std'  : [0.229, 0.224, 0.225],
    'label_names'   : {0: 'Normal', 1: 'Glaucoma'},
}

print('⚙️  Config chargée')
for k, v in CFG.items():
    print(f'   {k:<20} : {v}')

## Section 3 : Reconstruction de df_pairs

> Chaque session Kaggle est isolée — on reconstruit df_pairs ici  
> (même logique que 01_eda, 02_preprocessing, 03_fundus_model).

In [ ]:
# ── Lecture Excel + vote majoritaire ──────────────────────────────────────
df_raw = pd.read_excel(EXCEL_PATH, header=1)
df_raw.columns = [str(c).strip() for c in df_raw.columns]

glaucoma_cols = [df_raw.columns[2], df_raw.columns[4], df_raw.columns[6], df_raw.columns[8]]
image_col     = df_raw.columns[0]

def majority_vote(row):
    votes = []
    for col in glaucoma_cols:
        val = str(row[col]).lower().strip()
        if val in ['yes', 'oui', '1']:    votes.append(1)
        elif val in ['no', 'non', '0']:   votes.append(0)
        elif 'suspect' in val:            votes.append(0.5)
    if not votes: return np.nan
    return 1 if np.mean(votes) >= 0.5 else 0

df_raw['label']      = df_raw.apply(majority_vote, axis=1)
df_raw['label_name'] = df_raw['label'].map({1: 'Glaucoma', 0: 'Normal'})
df_raw = df_raw.dropna(subset=['label']).reset_index(drop=True)

# ── Index de tous les fichiers ─────────────────────────────────────────────
all_files = {}
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        all_files[f] = os.path.join(root, f)

# ── Construction des paires Fundus + OCT ──────────────────────────────────
records = []
for _, row in df_raw.iterrows():
    raw_name = str(row[image_col]).strip().strip("'\"*")
    if not raw_name.lower().endswith('.jpg'):
        raw_name += '.jpg'
    oct_name    = raw_name.replace('Color', 'B-scan')
    fundus_path = all_files.get(raw_name)
    oct_path    = all_files.get(oct_name)
    if fundus_path and oct_path:
        records.append({
            'fundus_path': fundus_path,
            'oct_path'   : oct_path,
            'label'      : int(row['label']),
            'label_name' : row['label_name'],
        })

df_pairs = pd.DataFrame(records)
print(f'✅ df_pairs reconstruit : {len(df_pairs)} paires')
print(df_pairs['label_name'].value_counts())
df_pairs.head(3)

## Section 4 : Dataset PyTorch & Transforms OCT

> **Différences vs Fundus :**
> - Chargement en **Grayscale → RGB** (OCT est monochromatique)
> - Pas de `HueSaturationValue` (image monochrome, pas de teinte à augmenter)
> - `CLAHE` plus fort (clip=3.0, p=0.4) — améliore le contraste des couches rétiniennes
> - Rotation réduite à ±10° (les B-scans OCT ont une orientation horizontale imposée)
> - `CoarseDropout` horizontal (hole_width_range plus large) — simule des artefacts de scan

In [ ]:
# ── Transforms OCT (Albumentations >= 1.4) ────────────────────────────────
def get_oct_train_transform(img_size=CFG['img_size']):
    """Augmentation pour images OCT B-scan (grayscale traité en RGB)."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        # Pas de VerticalFlip — les B-scans ont un sens anatomique haut/bas
        A.Affine(
            translate_percent={'x': (-0.08, 0.08), 'y': (-0.08, 0.08)},
            scale=(0.90, 1.10),
            rotate=(-10, 10),          # rotation réduite vs Fundus (±30°)
            border_mode=cv2.BORDER_REFLECT,
            p=0.6
        ),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.35, p=0.6),
        # Pas de HueSaturationValue — image monochrome
        A.CLAHE(clip_limit=3.0, p=0.4),   # plus fort que Fundus (2.0)
        A.GaussNoise(std_range=(0.02, 0.10), p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5)),
            A.MedianBlur(blur_limit=3),    # pas de MotionBlur pour OCT
        ], p=0.3),
        # CoarseDropout horizontal — simule artefacts de scan OCT
        A.CoarseDropout(
            num_holes_range=(1, 6),
            hole_height_range=(5, 15),
            hole_width_range=(10, 40),    # holes horizontaux plus larges
            fill=0,
            p=0.25
        ),
        A.Normalize(mean=CFG['imagenet_mean'], std=CFG['imagenet_std']),
        ToTensorV2(),
    ])


def get_oct_val_transform(img_size=CFG['img_size']):
    """Pas d'augmentation en validation — resize + normalize uniquement."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=CFG['imagenet_mean'], std=CFG['imagenet_std']),
        ToTensorV2(),
    ])


# ── Dataset PyTorch OCT ────────────────────────────────────────────────────
class OCTDataset(Dataset):
    """
    Dataset retournant uniquement l'image OCT B-scan et son label.
    Chargement : Grayscale → RGB (3 canaux identiques).
    Compatible avec df_pairs (colonne oct_path).
    """
    def __init__(self, df: pd.DataFrame, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        # Grayscale → RGB : 3 canaux identiques, compatible ResNet50
        img_np = np.array(
            Image.open(row['oct_path']).convert('L').convert('RGB'),
            dtype=np.uint8
        )
        tensor = self.transform(image=img_np)['image']
        label  = torch.tensor(int(row['label']), dtype=torch.long)
        return tensor, label


# ── Test rapide ─────────────────────────────────────────────────────────────
_ds = OCTDataset(df_pairs, get_oct_train_transform())
img, lbl = _ds[0]
print('✅ OCTDataset OK')
print(f'   Image shape  : {img.shape}  dtype={img.dtype}')
print(f'   Label        : {lbl.item()} ({CFG["label_names"][lbl.item()]})')
print(f'   Dataset len  : {len(_ds)}')
print(f'   Transforms   : {len(get_oct_train_transform().transforms)} (train)  /  {len(get_oct_val_transform().transforms)} (val)')

## Section 5 : Visualisation des Augmentations OCT

In [ ]:
def denormalize(tensor):
    mean = torch.tensor(CFG['imagenet_mean']).view(3, 1, 1)
    std  = torch.tensor(CFG['imagenet_std']).view(3, 1, 1)
    return (tensor * std + mean).permute(1, 2, 0).numpy().clip(0, 1)

# Prendre un exemple de chaque classe
sample_glaucoma = df_pairs[df_pairs['label'] == 1].iloc[0]
sample_normal   = df_pairs[df_pairs['label'] == 0].iloc[0]

N_AUG = 4
fig, axes = plt.subplots(2, N_AUG + 1, figsize=((N_AUG + 1) * 3.5, 7))
fig.suptitle('Augmentations OCT — Glaucome (haut) / Normal (bas)', fontsize=13, fontweight='bold')

tfm = get_oct_train_transform()

for row_i, (sample, title) in enumerate([
    (sample_glaucoma, 'Glaucome'),
    (sample_normal,   'Normal')
]):
    img_np = np.array(
        Image.open(sample['oct_path']).convert('L').convert('RGB'),
        dtype=np.uint8
    )
    orig = np.array(
        Image.open(sample['oct_path']).convert('L').convert('RGB').resize((CFG['img_size'], CFG['img_size']))
    )
    axes[row_i, 0].imshow(orig, cmap='gray')
    axes[row_i, 0].set_title(f'Original\n{title}', fontweight='bold', fontsize=9)
    axes[row_i, 0].axis('off')

    for j in range(N_AUG):
        aug = denormalize(tfm(image=img_np)['image'])
        axes[row_i, j+1].imshow(aug, cmap='gray')
        axes[row_i, j+1].set_title(f'Aug {j+1}', fontsize=9)
        axes[row_i, j+1].axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/oct_augmentations_viz.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('✅ Visualisation augmentations OCT sauvegardée')

## Section 6 : StratifiedKFold

In [ ]:
skf    = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
X      = df_pairs.index.values
y      = df_pairs['label'].values
folds  = list(skf.split(X, y))

print(f'✅ StratifiedKFold : {CFG["n_folds"]} folds (même seed={CFG["seed"]} que notebook 03)')
print(f'{"Fold":<6} {"Train":<8} {"Val":<6} {"Train G/N":<14} {"Val G/N"}')
print('-' * 50)
for i, (train_idx, val_idx) in enumerate(folds):
    y_tr = y[train_idx]
    y_va = y[val_idx]
    print(f'{i+1:<6} {len(train_idx):<8} {len(val_idx):<6} '
          f'{y_tr.sum()}/{len(y_tr)-y_tr.sum():<8} '
          f'{y_va.sum()}/{len(y_va)-y_va.sum()}')
print('\n⚠️  Les splits sont identiques au notebook 03 (même seed) → comparaison fair fold-à-fold')

## Section 7 : Architecture du Modèle

> Architecture **identique** au notebook 03.  
> Seules les images d'entrée changent (OCT vs Fundus).

In [ ]:
def build_oct_model() -> nn.Module:
    """
    ResNet50 préentraîné ImageNet avec tête de classification binaire.
    Architecture identique au modèle Fundus — seuls les poids entraînés diffèrent.

    Tête : Linear(2048→512) → BN → ReLU → Dropout(0.4) → Linear(512→1)
    Sortie : logit unique → BCEWithLogitsLoss
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    # Geler le backbone initialement (Phase A)
    for param in model.parameters():
        param.requires_grad = False

    # Remplacer la couche FC finale
    in_features = model.fc.in_features  # 2048
    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.4),
        nn.Linear(512, 1),
    )

    return model


_m = build_oct_model()
total_p     = sum(p.numel() for p in _m.parameters())
trainable_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)

print('🏗️  Modèle : ResNet50 + Tête binaire (OCT)')
print(f'   Paramètres totaux      : {total_p:,}')
print(f'   Paramètres entraînable : {trainable_p:,}  ← tête uniquement (Phase A)')
print(f'   Paramètres gelés       : {total_p - trainable_p:,}')
print()
print('📐 Tête de classification :')
print(_m.fc)
del _m

## Section 8 : Fonctions d'Entraînement

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, is_train: bool):
    """
    Une epoch complète (train ou val).
    Retourne : (avg_loss, accuracy)
    """
    model.train() if is_train else model.eval()

    total_loss = 0.0
    correct    = 0
    total      = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze(1)
            loss    = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds       = (torch.sigmoid(outputs) >= 0.5).long()
            correct    += (preds == labels.long()).sum().item()
            total      += images.size(0)

    return total_loss / total, correct / total


def evaluate_fold(model, loader, device):
    """
    Collecte labels, prédictions et probabilités sur un loader.
    Retourne : (y_true, y_pred, y_prob)
    """
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(device)
            outputs = model(images).squeeze(1)
            probs   = torch.sigmoid(outputs)
            preds   = (probs >= 0.5).long()

            all_labels.extend(labels.numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


print('✅ Fonctions run_epoch() et evaluate_fold() définies')

## Section 9 : Boucle K-Fold — Entraînement Complet

In [ ]:
# ── pos_weight pour déséquilibre de classes ────────────────────────────────
# Même ratio que notebook 03 : Normal=28, Glaucoma=20 → pos_weight=1.4
n_normal   = (y == 0).sum()
n_glaucoma = (y == 1).sum()
pos_weight = torch.tensor([n_normal / n_glaucoma], dtype=torch.float).to(DEVICE)
print(f'pos_weight = {pos_weight.item():.3f}  (Normal={n_normal} / Glaucoma={n_glaucoma})')

# ── Stockage des résultats ─────────────────────────────────────────────────
fold_metrics   = []
all_val_probs  = np.zeros(len(df_pairs))         # probas pour fusion notebook 05
all_val_labels = np.zeros(len(df_pairs), dtype=int)

print('\n🚀 Début de l\'entraînement K-Fold OCT')
print('=' * 72)

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    fold_num = fold_idx + 1
    print(f'\n╔══ FOLD {fold_num}/{CFG["n_folds"]} ══════════════════════════════════════════════╗')

    # ── DataLoaders ───────────────────────────────────────────────────────
    df_train = df_pairs.iloc[train_idx].reset_index(drop=True)
    df_val   = df_pairs.iloc[val_idx].reset_index(drop=True)

    ds_train = OCTDataset(df_train, get_oct_train_transform())
    ds_val   = OCTDataset(df_val,   get_oct_val_transform())

    train_loader = DataLoader(
        ds_train, batch_size=CFG['batch_size'], shuffle=True,
        num_workers=CFG['num_workers'], pin_memory=(DEVICE.type == 'cuda'), drop_last=False
    )
    val_loader = DataLoader(
        ds_val, batch_size=CFG['batch_size'], shuffle=False,
        num_workers=CFG['num_workers'], pin_memory=(DEVICE.type == 'cuda')
    )

    # ── Modèle + Loss + Optimizer + Scheduler ─────────────────────────────
    model     = build_oct_model().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.AdamW(
        model.fc.parameters(),
        lr=CFG['learning_rate'],
        weight_decay=CFG['weight_decay']
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min',
        factor=CFG['lr_factor'],
        patience=CFG['lr_patience'],
    )

    # ── Historique du fold ─────────────────────────────────────────────────
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    best_val_loss  = float('inf')
    patience_count = 0
    best_weights   = copy.deepcopy(model.state_dict())

    print(f'  {"Epoch":>5}  {"Train Loss":>10}  {"Val Loss":>10}  '
          f'{"Train Acc":>10}  {"Val Acc":>10}  {"LR":>10}')
    print('  ' + '-' * 68)

    start_time = time.time()

    for epoch in range(1, CFG['num_epochs'] + 1):

        # Phase B : dégeler le backbone après UNFREEZE_EPOCH
        if epoch == CFG['unfreeze_epoch'] + 1:
            print(f'\n  🔓 Epoch {epoch} : backbone dégelé — fine-tuning complet')
            for param in model.parameters():
                param.requires_grad = True
            optimizer.add_param_group({
                'params': [p for name, p in model.named_parameters() if 'fc' not in name],
                'lr': CFG['backbone_lr']
            })

        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, DEVICE, True)
        val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, optimizer, DEVICE, False)

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        flag = ''
        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            best_weights   = copy.deepcopy(model.state_dict())
            patience_count = 0
            flag           = '  ✅ best'
        else:
            patience_count += 1
            flag            = f'  ({patience_count}/{CFG["patience"]})'

        print(f'  {epoch:>5}  {train_loss:>10.4f}  {val_loss:>10.4f}  '
              f'{train_acc:>10.4f}  {val_acc:>10.4f}  {current_lr:>10.2e}{flag}')

        if patience_count >= CFG['patience']:
            print(f'\n  ⏹️  Early stopping à l\'epoch {epoch}')
            break

    elapsed = time.time() - start_time
    model.load_state_dict(best_weights)

    # ── Évaluation du fold ────────────────────────────────────────────────
    y_true, y_pred, y_prob = evaluate_fold(model, val_loader, DEVICE)

    acc     = accuracy_score(y_true, y_pred)
    f1      = f1_score(y_true, y_pred, zero_division=0)
    auc     = roc_auc_score(y_true, y_prob)
    cm      = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sens    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec    = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    fold_metrics.append({
        'fold': fold_num, 'accuracy': acc, 'f1': f1,
        'roc_auc': auc, 'sensitivity': sens, 'specificity': spec,
        'best_val_loss': best_val_loss, 'history': history
    })

    # Stocker les probas OOF pour le notebook 05 (Fusion)
    all_val_probs[val_idx]  = y_prob
    all_val_labels[val_idx] = y_true

    print(f'\n  📊 Fold {fold_num} — Résultats :')
    print(f'     Accuracy    : {acc*100:.2f}%')
    print(f'     F1 Score    : {f1:.4f}')
    print(f'     ROC-AUC     : {auc:.4f}')
    print(f'     Sensibilité : {sens*100:.2f}%  (détection Glaucome)')
    print(f'     Spécificité : {spec*100:.2f}%')
    print(f'     Durée       : {elapsed/60:.1f} min')
    print(f'╚═══════════════════════════════════════════════════════════════╝')

    # Sauvegarder le modèle du fold
    torch.save({
        'model_state_dict': model.state_dict(),
        'config'          : CFG,
        'fold'            : fold_num,
        'best_val_loss'   : best_val_loss,
        'metrics'         : fold_metrics[-1],
    }, f'{OUTPUT_DIR}/oct_model_fold{fold_num}.pth')
    print(f'  💾 Modèle sauvegardé → oct_model_fold{fold_num}.pth')

print('\n✅ Entraînement K-Fold OCT terminé')

## Section 10 : Résumé des Métriques K-Fold

In [ ]:
df_metrics = pd.DataFrame([{
    'Fold'        : m['fold'],
    'Accuracy'    : m['accuracy'],
    'F1 Score'    : m['f1'],
    'ROC-AUC'     : m['roc_auc'],
    'Sensibilité' : m['sensitivity'],
    'Spécificité' : m['specificity'],
} for m in fold_metrics])

mean_row = df_metrics.drop(columns='Fold').mean()
mean_row['Fold'] = 'Moyenne'
df_metrics = pd.concat([df_metrics, pd.DataFrame([mean_row])], ignore_index=True)

print('\n📊 Métriques OCT par fold')
print('=' * 70)
print(df_metrics.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('=' * 70)

best_fold_idx = np.argmax([m['roc_auc'] for m in fold_metrics])
best_fold     = fold_metrics[best_fold_idx]
print(f'\n🏆 Meilleur fold OCT : Fold {best_fold["fold"]} (ROC-AUC = {best_fold["roc_auc"]:.4f})')

# ── Comparaison Fundus vs OCT (si fundus_probabilities.npy disponible) ────
fundus_probs_path = Path(f'{OUTPUT_DIR}/fundus_probabilities.npy')
if fundus_probs_path.exists():
    fundus_probs = np.load(str(fundus_probs_path))
    fundus_auc   = roc_auc_score(all_val_labels, fundus_probs)
    oct_auc      = roc_auc_score(all_val_labels, all_val_probs)
    print(f'\n📊 Comparaison OOF AUC :')
    print(f'   Fundus ROC-AUC (moyen folds) : {np.mean([m["roc_auc"] for m in fold_metrics]):.4f}')
    print(f'   OCT    ROC-AUC (moyen folds) : {np.mean([m["roc_auc"] for m in fold_metrics]):.4f}')
    print(f'   Fundus AUC global OOF        : {fundus_auc:.4f}')
    print(f'   OCT    AUC global OOF        : {oct_auc:.4f}')
else:
    print('\n⚠️  fundus_probabilities.npy non trouvé — comparaison ignorée')
    print('   (Normal si outputs/ d\'une session précédente ne sont pas mountés)')

## Section 11 : Courbes d'Entraînement

In [ ]:
fig, axes = plt.subplots(CFG['n_folds'], 3, figsize=(18, CFG['n_folds'] * 4))
fig.suptitle('Courbes d\'entraînement — OCT ResNet50 (K-Fold)',
             fontsize=14, fontweight='bold')

for i, m in enumerate(fold_metrics):
    h   = m['history']
    eps = range(1, len(h['train_loss']) + 1)

    # Loss
    ax = axes[i, 0]
    ax.plot(eps, h['train_loss'], color=PALETTE['train'], linewidth=2,
            label='Train', marker='o', markersize=3)
    ax.plot(eps, h['val_loss'],   color=PALETTE['val'],   linewidth=2,
            label='Val',   marker='o', markersize=3)
    best_ep = np.argmin(h['val_loss']) + 1
    ax.axvline(best_ep, color='white', linestyle='--', linewidth=1.2,
               label=f'Best ep={best_ep}')
    ax.set_title(f'Fold {m["fold"]} — Loss'); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Accuracy
    ax = axes[i, 1]
    ax.plot(eps, [a*100 for a in h['train_acc']], color=PALETTE['train'],
            linewidth=2, label='Train', marker='o', markersize=3)
    ax.plot(eps, [a*100 for a in h['val_acc']],   color=PALETTE['val'],
            linewidth=2, label='Val',   marker='o', markersize=3)
    ax.axvline(best_ep, color='white', linestyle='--', linewidth=1.2)
    ax.set_title(f'Fold {m["fold"]} — Accuracy (%)')
    ax.set_xlabel('Epoch'); ax.set_ylim(0, 105)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Learning Rate
    ax = axes[i, 2]
    ax.plot(eps, h['lr'], color=PALETTE['neutral'], linewidth=2,
            marker='o', markersize=3)
    ax.set_title(f'Fold {m["fold"]} — Learning Rate')
    ax.set_xlabel('Epoch'); ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/oct_training_curves.png',
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('✅ Courbes OCT sauvegardées → oct_training_curves.png')

## Section 12 : Matrices de Confusion & ROC Curves

In [ ]:
fig, axes = plt.subplots(2, CFG['n_folds'], figsize=(CFG['n_folds'] * 4, 10))
fig.suptitle('Matrices de Confusion & ROC — OCT (K-Fold)',
             fontsize=13, fontweight='bold')

for i, (train_idx, val_idx) in enumerate(folds):
    m       = fold_metrics[i]
    model_o = build_oct_model().to(DEVICE)
    ckpt    = torch.load(
        f'{OUTPUT_DIR}/oct_model_fold{i+1}.pth',
        map_location=DEVICE,
        weights_only=False   # nécessaire pour charger le dict complet
    )
    model_o.load_state_dict(ckpt['model_state_dict'])

    ds_val     = OCTDataset(df_pairs.iloc[val_idx], get_oct_val_transform())
    val_loader = DataLoader(ds_val, batch_size=CFG['batch_size'], shuffle=False,
                            num_workers=CFG['num_workers'])
    y_true, y_pred, y_prob = evaluate_fold(model_o, val_loader, DEVICE)

    # ── Confusion Matrix ──────────────────────────────────────────────────
    ax = axes[0, i]
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',  # Blues pour OCT vs Reds pour Fundus
                xticklabels=['Normal', 'Glaucome'],
                yticklabels=['Normal', 'Glaucome'],
                ax=ax, linewidths=0.5,
                annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(f'Fold {i+1}\nACC={m["accuracy"]*100:.1f}%', fontsize=9)
    ax.set_xlabel('Prédit', fontsize=8)
    ax.set_ylabel('Réel', fontsize=8)

    # ── ROC Curve ─────────────────────────────────────────────────────────
    ax  = axes[1, i]
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val     = m['roc_auc']
    ax.plot(fpr, tpr, color=PALETTE['normal'], linewidth=2,  # vert pour OCT
            label=f'AUC={auc_val:.3f}')
    ax.plot([0,1],[0,1], color='gray', linestyle='--', linewidth=1.5)
    ax.fill_between(fpr, tpr, alpha=0.15, color=PALETTE['normal'])
    ax.set_title(f'Fold {i+1} — ROC', fontsize=9)
    ax.set_xlabel('FPR', fontsize=8); ax.set_ylabel('TPR', fontsize=8)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim(0,1); ax.set_ylim(0,1.02)

    del model_o

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/oct_evaluation.png',
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('✅ Évaluation OCT sauvegardée → oct_evaluation.png')

## Section 13 : Sauvegarde des Probabilités pour la Fusion (Notebook 05)

In [ ]:
# Sauvegarder les probabilités OOF — input du notebook 05
np.save(f'{OUTPUT_DIR}/oct_probabilities.npy', all_val_probs)
np.save(f'{OUTPUT_DIR}/oct_labels.npy',        all_val_labels)

print('💾 Probabilités OOF OCT sauvegardées :')
print(f'   oct_probabilities.npy  : shape={all_val_probs.shape}')
print(f'   oct_labels.npy         : shape={all_val_labels.shape}')
print(f'   Valeurs min/max        : [{all_val_probs.min():.4f}, {all_val_probs.max():.4f}]')

# Sauvegarder le meilleur modèle (fold avec meilleur ROC-AUC)
best_fold_num = fold_metrics[best_fold_idx]['fold']
import shutil
shutil.copy(
    f'{OUTPUT_DIR}/oct_model_fold{best_fold_num}.pth',
    f'{OUTPUT_DIR}/oct_model_best.pth'
)
print(f'\n💾 Meilleur modèle OCT (fold {best_fold_num}) → oct_model_best.pth')

# ── Vérification cohérence avec fundus_labels.npy ─────────────────────────
fundus_labels_path = Path(f'{OUTPUT_DIR}/fundus_labels.npy')
if fundus_labels_path.exists():
    fundus_labels = np.load(str(fundus_labels_path))
    if np.array_equal(fundus_labels, all_val_labels):
        print('\n✅ Labels OCT == Labels Fundus — cohérence parfaite pour fusion')
    else:
        print('\n⚠️  Labels OCT != Labels Fundus — vérifier le seed ou l\'ordre des données!')
else:
    print('\n⚠️  fundus_labels.npy non trouvé — vérification impossible')

## Section 14 : Validation Finale

In [ ]:
mean_acc  = np.mean([m['accuracy']    for m in fold_metrics])
mean_auc  = np.mean([m['roc_auc']     for m in fold_metrics])
mean_sens = np.mean([m['sensitivity'] for m in fold_metrics])
mean_f1   = np.mean([m['f1']          for m in fold_metrics])
mean_spec = np.mean([m['specificity'] for m in fold_metrics])

assertions = [
    (mean_acc  > 0.55, f'Accuracy moyenne > 55%    (obtenu {mean_acc*100:.1f}%)'),
    (mean_auc  > 0.60, f'ROC-AUC moyen > 0.60      (obtenu {mean_auc:.3f})'),
    (Path(f'{OUTPUT_DIR}/oct_probabilities.npy').exists(), 'oct_probabilities.npy sauvegardé'),
    (Path(f'{OUTPUT_DIR}/oct_model_best.pth').exists(),    'oct_model_best.pth sauvegardé'),
    (Path(f'{OUTPUT_DIR}/oct_training_curves.png').exists(),'training curves sauvegardées'),
    (Path(f'{OUTPUT_DIR}/oct_evaluation.png').exists(),    'evaluation plots sauvegardés'),
    (Path(f'{OUTPUT_DIR}/oct_labels.npy').exists(),        'oct_labels.npy sauvegardé'),
]

all_passed = True
for condition, name in assertions:
    status = '✅' if condition else '⚠️ '
    print(f'   {status}  {name}')
    if not condition:
        all_passed = False

print()
if all_passed:
    print('🎉 TOUTES LES ASSERTIONS PASSÉES — Notebook 04 terminé !')
else:
    print('⚠️  Certaines métriques sous le seuil (dataset de 48 images — normal)')

print()
print('=' * 55)
print('  📊 RÉSULTATS FINAUX — OCT ResNet50 (K=5)')
print('=' * 55)
print(f'  Accuracy moyenne    : {mean_acc*100:.2f}%')
print(f'  F1 Score moyen      : {mean_f1:.4f}')
print(f'  ROC-AUC moyen       : {mean_auc:.4f}')
print(f'  Sensibilité moyenne : {mean_sens*100:.2f}%')
print(f'  Spécificité moyenne : {mean_spec*100:.2f}%')
print()
print('📦 Fichiers générés :')
print(f'   oct_model_fold{{1..{CFG["n_folds"]}}}.pth   → modèles par fold')
print(f'   oct_model_best.pth              → meilleur fold (fold {best_fold_num})')
print(f'   oct_probabilities.npy           → input notebook 05 (Fusion)')
print(f'   oct_labels.npy                  → vérification')
print(f'   oct_training_curves.png         → courbes loss/acc/lr')
print(f'   oct_evaluation.png              → CM + ROC par fold')
print(f'   oct_augmentations_viz.png       → visualisation augmentations')
print()
print('➡️  Prochain notebook : 05_fusion.ipynb')
print('   Inputs nécessaires :')
print('   • fundus_probabilities.npy  (notebook 03)')
print('   • oct_probabilities.npy     (ce notebook)')
print('   • fundus_labels.npy ou oct_labels.npy (identiques)')